# KKBox Churn Prediction - Exploratory Data Analysis

This notebook documents the data-understanding phase for the KKBox churn project. It uses only configured paths and persisted pipeline artifacts, so the analysis can be rerun without reading or mutating `data/raw/`.

Business objective: identify subscriber patterns associated with churn so downstream modeling can support retention prioritization.

## 1. Notebook Contract

Inputs:
- `config/config.yaml`
- `data/interim/*.parquet` produced by `python -m src.data.run_ingestion`
- `data/processed/feature_frame.parquet` when feature engineering has already been run

Outputs:
- Tables and charts rendered in the notebook
- Optional figures under the configured `reports/figures` directory

This notebook is diagnostic only; it does not train models or write to `data/raw/`.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.engineer import build_feature_frame
from src.utils.config import get_path, get_value, load_config

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(CONFIG_PATH)

interim_dir = get_path(config, "interim_dir", base_dir=PROJECT_ROOT)
processed_dir = get_path(config, "processed_dir", base_dir=PROJECT_ROOT)
figures_dir = get_path(config, "figures_dir", base_dir=PROJECT_ROOT)
figures_dir.mkdir(parents=True, exist_ok=True)

target_col = get_value(config, "project", "target_col")
id_col = get_value(config, "project", "id_col")
preview_rows = int(get_value(config, "eda", "preview_rows"))
top_categories = int(get_value(config, "eda", "top_categories"))
correlation_min_abs = float(get_value(config, "eda", "correlation_min_abs"))
figure_dpi = int(get_value(config, "eda", "figure_dpi"))
random_state = int(get_value(config, "project", "random_state"))

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Interim data: {interim_dir}")
print(f"Processed data: {processed_dir}")

## 2. Load Pipeline Artifacts

The EDA starts from `data/interim/` because ingestion already validates raw schemas, applies the temporal cutoff, and writes compact Parquet artifacts. That keeps exploration aligned with the production pipeline.

In [ ]:
required_interim_files = {
    "train": interim_dir / "train.parquet",
    "members": interim_dir / "members.parquet",
    "transactions": interim_dir / "transactions_summary.parquet",
    "user_logs": interim_dir / "user_logs_summary.parquet",
    "modeling_frame": interim_dir / "modeling_frame.parquet",
}

missing_files = [str(path) for path in required_interim_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Run `python -m src.data.run_ingestion` before this notebook. Missing: "
        + ", ".join(missing_files)
    )

tables = {name: pd.read_parquet(path) for name, path in required_interim_files.items()}
feature_frame_path = processed_dir / str(get_value(config, "feature_engineering", "output_file"))

if feature_frame_path.exists():
    feature_frame = pd.read_parquet(feature_frame_path)
else:
    feature_frame, _ = build_feature_frame(tables["modeling_frame"], config)

print("Loaded tables:")
for name, frame in tables.items():
    print(f"- {name}: {frame.shape[0]:,} rows x {frame.shape[1]:,} columns")
print(f"- feature_frame: {feature_frame.shape[0]:,} rows x {feature_frame.shape[1]:,} columns")

## 3. Schema and Grain Audit

This section checks row counts, column types, duplicate business keys, and target availability. These checks catch common production issues before model training.

In [ ]:
def audit_table(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "table": [name],
        "rows": [len(frame)],
        "columns": [frame.shape[1]],
        "duplicate_rows": [int(frame.duplicated().sum())],
        "duplicate_ids": [int(frame.duplicated(subset=[id_col]).sum()) if id_col in frame.columns else np.nan],
        "missing_target": [int(frame[target_col].isna().sum()) if target_col in frame.columns else np.nan],
    })

audit = pd.concat([audit_table(frame, name) for name, frame in tables.items()], ignore_index=True)
display(audit)

display(feature_frame.head(preview_rows))
display(feature_frame.dtypes.rename("dtype").reset_index().rename(columns={"index": "column"}))

## 4. Target Distribution

Churn is expected to be imbalanced. Measuring prevalence early informs metric selection, thresholding, and retention campaign capacity planning.

In [ ]:
target_counts = feature_frame[target_col].value_counts(dropna=False).rename_axis(target_col).reset_index(name="count")
target_counts["share"] = target_counts["count"] / target_counts["count"].sum()
display(target_counts)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=target_counts, x=target_col, y="share", ax=ax, color="#4C78A8")
ax.set_title("Churn prevalence")
ax.set_ylabel("Share of users")
ax.set_xlabel("Churn label")
plt.tight_layout()
plt.show()

## 5. Missingness Profile

Missing values can be direct behavioral signal in subscription products. The goal here is to distinguish harmless sparsity from fields that need explicit missing indicators or upstream validation.

In [ ]:
missing_summary = (
    feature_frame.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_rate=lambda frame: frame["missing_count"] / len(feature_frame))
    .query("missing_count > 0")
    .sort_values("missing_rate", ascending=False)
)

display(missing_summary.head(top_categories))

if not missing_summary.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    missing_summary.head(top_categories)["missing_rate"].sort_values().plot(kind="barh", ax=ax, color="#F58518")
    ax.set_title("Highest missing-value rates")
    ax.set_xlabel("Missing rate")
    plt.tight_layout()
    plt.show()

## 6. Numeric Feature Distributions

The distribution review focuses on high-signal behavioral and monetary fields. Skewed variables should be transformed in feature engineering rather than patched in the notebook.

In [ ]:
candidate_numeric_features = [
    "total_spend",
    "mean_spend",
    "cancel_rate",
    "auto_renew_rate",
    "active_days",
    "total_secs",
    "mean_completion_rate",
    "days_since_last_transaction",
    "days_since_last_log",
]

numeric_features = [column for column in candidate_numeric_features if column in feature_frame.columns]
summary = feature_frame[numeric_features].describe(percentiles=[0.25, 0.5, 0.75]).T
display(summary)

for column in numeric_features:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    sns.histplot(feature_frame[column].dropna(), ax=axes[0], color="#4C78A8", kde=True)
    sns.boxplot(x=feature_frame[column], ax=axes[1], color="#72B7B2")
    axes[0].set_title(f"Distribution: {column}")
    axes[1].set_title(f"Outliers: {column}")
    plt.tight_layout()
    plt.show()

## 7. Categorical Feature Review

Categorical review checks class balance and churn lift across user attributes. Rare categories should be handled by the preprocessing layer rather than manually collapsed here.

In [ ]:
candidate_categorical_features = ["gender_clean", "age_band", "city", "registered_via"]
categorical_features = [column for column in candidate_categorical_features if column in feature_frame.columns]

for column in categorical_features:
    grouped = (
        feature_frame.groupby(column, dropna=False)[target_col]
        .agg(count="size", churn_rate="mean")
        .sort_values("count", ascending=False)
        .head(top_categories)
    )
    display(grouped)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    grouped["count"].sort_values().plot(kind="barh", ax=axes[0], color="#4C78A8")
    grouped["churn_rate"].sort_values().plot(kind="barh", ax=axes[1], color="#E45756")
    axes[0].set_title(f"Top categories by volume: {column}")
    axes[1].set_title(f"Churn rate by category: {column}")
    plt.tight_layout()
    plt.show()

## 8. Feature-to-Target Relationships

These checks identify candidate churn drivers and potential leakage. Strong relationships are not automatically bad, but they must be explainable from pre-cutoff behavior.

In [ ]:
numeric_frame = feature_frame.select_dtypes(include=[np.number])
correlations = (
    numeric_frame.corr(numeric_only=True)[target_col]
    .drop(labels=[target_col], errors="ignore")
    .dropna()
    .sort_values(key=lambda values: values.abs(), ascending=False)
)

display(correlations[correlations.abs() >= correlation_min_abs].to_frame("correlation_with_churn"))

selected_relationships = [column for column in numeric_features if column in correlations.index]
for column in selected_relationships:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.boxplot(data=feature_frame, x=target_col, y=column, ax=ax, color="#72B7B2")
    ax.set_title(f"{column} by churn label")
    plt.tight_layout()
    plt.show()

## 9. Temporal Safety Checks

Production churn modeling must avoid using events from the churn observation window. This section verifies that engineered recency features are anchored to the configured cutoff date.

In [ ]:
cutoff_date = pd.Timestamp(get_value(config, "feature_engineering", "cutoff_date"))
date_columns = [
    column for column in ["last_transaction_date", "last_log_date", "analysis_reference_date"]
    if column in feature_frame.columns
]

temporal_audit = []
for column in date_columns:
    values = pd.to_datetime(feature_frame[column], errors="coerce")
    temporal_audit.append({
        "column": column,
        "min": values.min(),
        "max": values.max(),
        "rows_after_cutoff": int((values > cutoff_date).sum()) if column != "analysis_reference_date" else 0,
    })

display(pd.DataFrame(temporal_audit))

## 10. Modeling Implications

The EDA should end with decisions that are useful for the pipeline, not isolated charts.

In [ ]:
findings = []
churn_rate = float(feature_frame[target_col].mean())
findings.append(f"Churn prevalence is {churn_rate:.2%}; validation should prioritize ranking metrics such as AUC-PR.")

if not missing_summary.empty:
    findings.append(f"Highest missingness field is {missing_summary.index[0]} at {missing_summary.iloc[0]['missing_rate']:.2%}.")

if not correlations.empty:
    top_feature = correlations.index[0]
    findings.append(f"Strongest numeric association with churn is {top_feature} ({correlations.iloc[0]:.3f}).")

for item in findings:
    print(f"- {item}")